# Accessing Data

In [1]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

/home/565/pv3484/aus_substation_electricity
/home/565/pv3484/aus_substation_electricity


In [2]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

processing nsw substations for ['ausgrid'] from None to None
ausgrid
following columns in demand are not in info index:
['MT_HU', 'SI_NO']
removing these columns from demand
number of substations in ausgrid substation info: 134
number of substations in ausgrid substation data: 132
following sites match selection criteria:
               energy_asset          Name  Area  Dwellings  Persons  Residential  Commercial  Industrial  Primary Production  Education  \
ID                                                                                                                                        
BLAKE         AG_BLAKEHURST    Blakehurst     7      10081    28521        0.850       0.005       0.021               0.000      0.022   
PUNCH          AG_PUNCHBOWL     Punchbowl     9      17514    50395        0.826       0.048       0.038               0.000      0.025   
MEADO         AG_MEADOWBANK    Meadowbank    15      22420    56948        0.825       0.023       0.015               0

## Holiday Function

In [3]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

#Monarch's Birthday
def second_monday_of_june(y):
    """Return the date of the second Monday in June for year y."""
    june = pd.date_range(start=f"{y}-06-01", end=f"{y}-06-30", freq="D")
    mondays = june[june.weekday == 0]   # Monday = 0
    return mondays[1]                   # second Monday



# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Monarch's Birthday": lambda y: second_monday_of_june(y),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}


In [16]:
print(HOLIDAYS_VIC)


{"New Year's Day": <function <lambda> at 0x14beb91d7ba0>, 'Australia Day': <function <lambda> at 0x14beb91d7740>, 'Good Friday': <function <lambda> at 0x14beb91d7420>, 'Easter Saturday': <function <lambda> at 0x14beb91d74c0>, 'Easter Sunday': <function <lambda> at 0x14beb91d60c0>, 'Easter Monday': <function <lambda> at 0x14beeecd63e0>, 'ANZAC Day': <function <lambda> at 0x14beeecd68e0>, "Monarch's Birthday": <function <lambda> at 0x14beeecd6660>, 'Christmas Day': <function <lambda> at 0x14beeecd6b60>, 'Boxing Day': <function <lambda> at 0x14beb8f740e0>}


# Creating new csv file
- One row per day
- 30 days before the holiday
- the holiday itself
- 30 days after the holiday
- So 61 rows per holiday × year × station
- also ensures new year's day and christmas day include any days within the 30 +/- days that are not in the same year
- includes columns defining what the day's name is (Monday, Tuesday etc), and if it's a weekend (True/False)
----------------------------------
- For each day, the function will compute:
- Block‑level mean, standard deviation, and variance of the relative‑rank values for that day’s 24‑hour profile.
---------------------------------
- Using new time blocks:
- 04–10
- 10–15
- 15–20
- 20–24
- 00–04
---------------------------------
- Using your 2‑year forward‑looking ranking window (Y + Y+1)
- metadata integrated (station id, station name, residential fraction, industrial fraction, dwellings, persons)

In [7]:
import pandas as pd
import numpy as np
import os


def compute_two_year_daily_relative_rank_csv(
    demand,
    holiday_lib,
    info,
    window_days=30,
    blocks=None,
    out_csv="full_nsw_relative_rank.csv"
):
    """
    Computes daily block-level mean, std, and variance of relative ranks
    for each day in ±window_days around each holiday, for each year and station.

    Ranking pool = all hourly data from year Y and Y+1.
    Window extraction = ±window_days around holiday, with spillover allowed
    into previous/next years.

    Output = one row per station × holiday × year × day.
    """

    if blocks is None:
        raise ValueError("You must supply a dictionary of time blocks.")

    # Ensure datetime index
    demand.index = pd.to_datetime(demand.index)

    # Hourly mean demand
    hourly = demand.resample("h").mean()

    # All station columns
    stations = [c for c in hourly.columns if c not in ["date", "hour"]]

    rows = []

    # ---------------------------------------------------------
    # LOOP HOLIDAYS
    # ---------------------------------------------------------
    for holiday_name, holiday_func in holiday_lib.items():

        for year in range(2004, 2018):

            ref_date = holiday_func(year)

            # ---------------------------------------------------------
            # EXPANDED POOL FOR WINDOW EXTRACTION
            # ---------------------------------------------------------
            pool_start = pd.Timestamp(f"{year}-01-01") - pd.Timedelta(days=31)
            pool_end   = pd.Timestamp(f"{year+1}-12-31") + pd.Timedelta(days=31)

            expanded_pool = hourly.loc[pool_start:pool_end]
            if expanded_pool.empty:
                continue

            # ---------------------------------------------------------
            # EXTRACT ±window_days AROUND HOLIDAY
            # ---------------------------------------------------------
            win_start = ref_date - pd.Timedelta(days=window_days)
            win_end   = ref_date + pd.Timedelta(days=window_days)

            window = expanded_pool.loc[win_start:win_end].copy()
            if window.empty:
                continue

            # Remove the holiday itself from the baseline window
           # window = window[window.index.date != ref_date.date()]

            window["date"] = window.index.date
            window["hour"] = window.index.hour

            # ---------------------------------------------------------
            # TRUE RANKING POOL (Y + Y+1 ONLY)
            # ---------------------------------------------------------
            rank_start = pd.Timestamp(f"{year}-01-01")
            rank_end   = pd.Timestamp(f"{year+1}-12-31")

            ranking_pool = hourly.loc[rank_start:rank_end]

            # ---------------------------------------------------------
            # LOOP STATIONS
            # ---------------------------------------------------------
            for station in stations:

                if station not in window.columns:
                    continue

                # Copy window for this station
                w = window.copy()

                # ---------------------------------------------------------
                # RELATIVE RANK PER HOUR (within Y + Y+1 only)
                # ---------------------------------------------------------
                rp = ranking_pool[[station]].copy()
                rp["date"] = rp.index.date
                rp["hour"] = rp.index.hour

                rp["rank"] = rp.groupby("hour")[station].rank(method="average")
                n_days = rp.groupby("hour")["date"].transform("nunique")
                rp["relative_rank"] = rp["rank"] / n_days

                # Merge relative ranks back into window
                w = w.merge(
                    rp[["relative_rank", "hour", "date"]],
                    on=["hour", "date"],
                    how="left"
                )

                # ---------------------------------------------------------
                # METADATA
                # ---------------------------------------------------------
                station_code = station
                station_name = info.loc[station_code, "Name"]
                residential  = info.loc[station_code, "Residential"]
                dwellings    = info.loc[station_code, "Dwellings"]
                persons      = info.loc[station_code, "Persons"]
                industrial   = info.loc[station_code, "Industrial"]

                # ---------------------------------------------------------
                # DAILY BLOCK STATISTICS
                # ---------------------------------------------------------
                for day in sorted(w["date"].unique()):

                    mask = w["date"] == day
                    day_rr = w.loc[mask, "relative_rank"]

                    # ---- FIX: always include the holiday day ----
                    is_holiday_day = (pd.Timestamp(day).date() == ref_date.date())

                    # baseline days must have ≥18 hours; holiday day is always kept
                    if (day_rr.shape[0] < 18) and (not is_holiday_day):
                        continue

                    # Reindex to hour 0–23
                    day_rr.index = w.loc[mask, "hour"]

                    # ---- weekday name + weekend flag ----
                    weekday_name = pd.Timestamp(day).day_name()
                    is_weekend = weekday_name in ["Saturday", "Sunday"]

                    block_stats = {}
                    for block_name, hours in blocks.items():
                        vals = day_rr.loc[list(hours)]
                        block_stats[f"{block_name}_mean"] = vals.mean()
                        block_stats[f"{block_name}_std"]  = vals.std()
                        block_stats[f"{block_name}_var"]  = vals.var()

                    rows.append({
                        "station_code": station_code,
                        "station_name": station_name,
                        "date": pd.Timestamp(day),
                        "weekday_name": weekday_name,
                        "is_weekend": is_weekend,
                        "year": year,
                        "holiday": holiday_name,
                        "is_holiday": is_holiday_day,
                        **block_stats,
                        "residential": residential,
                        "industrial": industrial,
                        "persons": persons,
                        "dwellings": dwellings,
                    })

    # ---------------------------------------------------------
    # BUILD FINAL DATAFRAME
    # ---------------------------------------------------------
    df = pd.DataFrame(rows).round(4)

    # ---------------------------------------------------------
    # REORDER COLUMNS
    # ---------------------------------------------------------
    block_cols = [c for c in df.columns if any(s in c for s in ["_mean", "_std", "_var"])]

    ordered_cols = (
        ["station_code", "station_name", "date", "weekday_name", "is_weekend",
         "year", "holiday", "is_holiday"]
        + block_cols
        + ["residential", "industrial", "persons", "dwellings"]
    )

    df = df[ordered_cols]

    # ---------------------------------------------------------
    # SAVE
    # ---------------------------------------------------------
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    df.to_csv(out_csv, index=False)

    return df


In [8]:
blocks = {
    "04_10": range(4, 10),
    "10_15": range(10, 15),
    "15_20": range(15, 20),
    "20_24": range(20, 24),
    "00_04": range(0, 4),
}


In [9]:
df = compute_two_year_daily_relative_rank_csv(
    demand=demand,
    holiday_lib=HOLIDAYS_VIC,
    info=info,
    blocks=blocks,
    out_csv="/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv"
)
